# Lab23: CIFAR-10 Image Classification with HoG + SVM vs. a Simple CNN

This lab compares a **classical computer vision pipeline** with a **deep learning pipeline** on the same image classification problem.

You will:
- extract **Histogram of Oriented Gradients (HoG)** features from CIFAR-10 images,
- train an **SVM** classifier on those features,
- build and train a **small Convolutional Neural Network (CNN)**,
- compare the two methods using accuracy, confusion matrices, and training time.

**Notes**
- The code is written to be simple and easy to follow.
- To keep execution practical on a CPU, the notebook uses a subset of CIFAR-10 by default.
- The notebook is ready to compute the results when you run it. Exact numbers may vary slightly by hardware, random seed, and number of epochs.


## Task 1 — Import the required libraries and define the experiment settings

Use Python libraries for:
1. loading CIFAR-10,
2. plotting images,
3. extracting HoG features,
4. training an SVM classifier,
5. building and training a CNN,
6. evaluating and comparing both approaches.

Keep the experiment reproducible by setting random seeds.


In [ ]:
import time
import random
import numpy as np
import matplotlib.pyplot as plt

from skimage.color import rgb2gray
from skimage.feature import hog

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Keep the experiment light enough for a normal CPU
TRAIN_SAMPLES = 10000
TEST_SAMPLES = 2000
BATCH_SIZE = 128
EPOCHS = 5

class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]


## Task 2 — Load CIFAR-10 and inspect the dataset

1. Load the training and test sets of CIFAR-10.
2. Create a smaller subset for faster experimentation.
3. Display a few sample images with their class labels.

Why is it useful to inspect a few examples before training a model?


In [ ]:
# For the CNN we keep images as tensors in [0, 1]
tensor_transform = transforms.ToTensor()

train_dataset_full = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=tensor_transform
)
test_dataset_full = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=tensor_transform
)

train_indices = list(range(TRAIN_SAMPLES))
test_indices = list(range(TEST_SAMPLES))

train_dataset = Subset(train_dataset_full, train_indices)
test_dataset = Subset(test_dataset_full, test_indices)

print("Training subset:", len(train_dataset))
print("Test subset:", len(test_dataset))

# Show a few examples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img_tensor, label = train_dataset[i]
    img_np = np.transpose(img_tensor.numpy(), (1, 2, 0))
    ax.imshow(img_np)
    ax.set_title(class_names[label], fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()


## Task 3 — Prepare NumPy arrays for the HoG + SVM pipeline

1. Convert the selected CIFAR-10 images into NumPy arrays.
2. Convert color images to grayscale for HoG extraction.
3. Prepare the target labels.

Why might classical feature extraction use grayscale images?


In [ ]:
def subset_to_numpy(subset):
    images = []
    labels = []
    for img_tensor, label in subset:
        img_np = np.transpose(img_tensor.numpy(), (1, 2, 0))  # HWC
        images.append(img_np)
        labels.append(label)
    return np.array(images), np.array(labels)

X_train_img, y_train = subset_to_numpy(train_dataset)
X_test_img, y_test = subset_to_numpy(test_dataset)

print("X_train_img shape:", X_train_img.shape)
print("y_train shape:", y_train.shape)
print("X_test_img shape:", X_test_img.shape)
print("y_test shape:", y_test.shape)


## Task 4 — Extract HoG features from the images

Use the `hog` function from `skimage.feature` to extract edge and gradient information from each image.

Suggested settings:
- orientations = 9
- pixels_per_cell = (4, 4)
- cells_per_block = (2, 2)

After feature extraction, report the final feature vector size.


In [ ]:
def extract_hog_features(images):
    features = []
    for img in images:
        gray = rgb2gray(img)
        feat = hog(
            gray,
            orientations=9,
            pixels_per_cell=(4, 4),
            cells_per_block=(2, 2),
            block_norm="L2-Hys"
        )
        features.append(feat)
    return np.array(features)

start_hog = time.time()
X_train_hog = extract_hog_features(X_train_img)
X_test_hog = extract_hog_features(X_test_img)
hog_time = time.time() - start_hog

print("HoG extraction time (seconds):", round(hog_time, 2))
print("HoG train feature shape:", X_train_hog.shape)
print("HoG test feature shape:", X_test_hog.shape)


## Task 5 — Train an SVM classifier on top of HoG features

1. Standardize the HoG feature vectors.
2. Train a simple linear SVM classifier.
3. Measure the training time.
4. Evaluate the model on the test subset.

Why is feature scaling useful before training an SVM?


In [ ]:
scaler = StandardScaler()
X_train_hog_scaled = scaler.fit_transform(X_train_hog)
X_test_hog_scaled = scaler.transform(X_test_hog)

svm_model = LinearSVC(random_state=SEED, max_iter=5000)

start_svm = time.time()
svm_model.fit(X_train_hog_scaled, y_train)
svm_train_time = time.time() - start_svm

svm_preds = svm_model.predict(X_test_hog_scaled)
svm_acc = accuracy_score(y_test, svm_preds)

print("SVM training time (seconds):", round(svm_train_time, 2))
print("HoG + SVM accuracy:", round(svm_acc, 4))
print()
print(classification_report(y_test, svm_preds, target_names=class_names))


## Task 6 — Visualize the confusion matrix for HoG + SVM

Create and display a confusion matrix.

Which classes does the HoG + SVM model confuse most often?


In [ ]:
svm_cm = confusion_matrix(y_test, svm_preds)

plt.figure(figsize=(8, 6))
plt.imshow(svm_cm, interpolation="nearest", cmap="Blues")
plt.title("Confusion Matrix — HoG + SVM")
plt.colorbar()
plt.xticks(range(len(class_names)), class_names, rotation=45, ha="right")
plt.yticks(range(len(class_names)), class_names)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()


## Task 7 — Build a simple CNN for CIFAR-10 classification

Construct a small CNN with only a few layers. A simple design is enough:
- convolution + ReLU,
- max pooling,
- convolution + ReLU,
- max pooling,
- fully connected layers.

Keep the model small and readable.


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 32x32 -> 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 16x16 -> 8x8

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)    # 8x8 -> 4x4
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

cnn_model = SimpleCNN().to(device)
print(cnn_model)


## Task 8 — Train the CNN model

1. Create data loaders for the same train and test subsets.
2. Use cross-entropy loss and Adam optimizer.
3. Train the CNN for a few epochs.
4. Print the training loss after each epoch.

How does end-to-end feature learning differ from hand-crafted feature extraction?


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

start_cnn = time.time()

for epoch in range(EPOCHS):
    cnn_model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = cnn_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {epoch_loss:.4f}")

cnn_train_time = time.time() - start_cnn
print("CNN training time (seconds):", round(cnn_train_time, 2))


## Task 9 — Evaluate the CNN on the test set

1. Compute the test accuracy.
2. Print a classification report.
3. Build the confusion matrix.

Does the CNN perform better than the HoG + SVM pipeline on CIFAR-10?


In [ ]:
cnn_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = cnn_model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

cnn_acc = accuracy_score(all_labels, all_preds)
cnn_cm = confusion_matrix(all_labels, all_preds)

print("CNN accuracy:", round(cnn_acc, 4))
print()
print(classification_report(all_labels, all_preds, target_names=class_names))


## Task 10 — Visualize the confusion matrix for the CNN

Plot the confusion matrix and inspect which classes remain difficult.

Which classes improved the most compared with HoG + SVM?


In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(cnn_cm, interpolation="nearest", cmap="Blues")
plt.title("Confusion Matrix — Simple CNN")
plt.colorbar()
plt.xticks(range(len(class_names)), class_names, rotation=45, ha="right")
plt.yticks(range(len(class_names)), class_names)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()


## Task 11 — Compare HoG + SVM and CNN results

Create a small comparison table with:
- test accuracy,
- training time,
- feature extraction time (for HoG),
- short conclusion.

Then answer:
1. Which method gave the better accuracy?
2. Which method was easier to interpret?
3. Which method would you choose for small images like CIFAR-10, and why?


In [ ]:
comparison = {
    "Model": ["HoG + SVM", "Simple CNN"],
    "Accuracy": [svm_acc, cnn_acc],
    "Training_time_sec": [svm_train_time, cnn_train_time],
    "Extra_feature_time_sec": [hog_time, 0.0]
}

import pandas as pd
comparison_df = pd.DataFrame(comparison)
comparison_df["Accuracy"] = comparison_df["Accuracy"].round(4)
comparison_df["Training_time_sec"] = comparison_df["Training_time_sec"].round(2)
comparison_df["Extra_feature_time_sec"] = comparison_df["Extra_feature_time_sec"].round(2)

comparison_df


## Task 12 — Extension questions

Try one or more of the following:
1. Increase the number of CNN epochs and observe the effect.
2. Add data augmentation for the CNN.
3. Replace `LinearSVC` with another classifier.
4. Change the HoG parameters and check whether the SVM improves.
5. Add one more convolutional layer to the CNN.

These are optional extensions for deeper exploration.


## Result note

This notebook is designed to **compute the results when run**.  
Because accuracy depends on the exact subset, hardware, random seed, and number of epochs, your final numbers may vary slightly.

In most runs on CIFAR-10, a simple CNN typically performs better than a HoG + SVM pipeline because the CNN learns task-specific image features directly from data.
